# Road Damage Detection Using CNNs

**Phase 2 - Proposal and Code Implementation - Ravi Ansh**

This notebook trains a custom CNN and a MobileNetV2 transfer learning model for road damage image classification.

Recommended Kaggle dataset: `sabidrahman/pothole-cracks-and-openmanhole`.

Classes: pothole, crack, open manhole.

In [ ]:
# ============================================================
# 1. Imports and Configuration
# ============================================================

import os
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 5

np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = Path('/kaggle/working/outputs') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('TensorFlow version:', tf.__version__)
print('Output folder:', OUTPUT_DIR)

## 2. Dataset Path Detection

Add the Kaggle dataset using **Add Input**. If the folder name is different, the code below searches `/kaggle/input` automatically.

In [ ]:
# ============================================================
# 2. Auto-detect image dataset folder
# ============================================================

SEARCH_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
print('Search root:', SEARCH_ROOT)

image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Print available input folders
if SEARCH_ROOT.exists():
    print('Available folders:', [p.name for p in SEARCH_ROOT.iterdir() if p.is_dir()])

# Find candidate folders that contain class subfolders with images
candidates = []
for root, dirs, files in os.walk(SEARCH_ROOT):
    root_path = Path(root)
    subdirs = [root_path / d for d in dirs]
    image_subdir_count = 0
    total_images = 0
    for sd in subdirs:
        try:
            imgs = [f for f in sd.iterdir() if f.suffix.lower() in image_exts]
            if len(imgs) > 0:
                image_subdir_count += 1
                total_images += len(imgs)
        except Exception:
            pass
    if image_subdir_count >= 2:
        candidates.append((str(root_path), image_subdir_count, total_images))

print('Candidate class-folder datasets:')
for c in candidates[:20]:
    print(c)

if len(candidates) == 0:
    raise FileNotFoundError(
        'No class-folder image dataset found. In Kaggle, click Add Input and add '
        'sabidrahman/pothole-cracks-and-openmanhole, then rerun this cell.'
    )

# Choose candidate with most images
DATASET_DIR = Path(sorted(candidates, key=lambda x: x[2], reverse=True)[0][0])
print('Using DATASET_DIR:', DATASET_DIR)
print('Class folders:', [p.name for p in DATASET_DIR.iterdir() if p.is_dir()])

## 3. Load Dataset

The code supports datasets with separate `train` and `valid/validation/test` folders, or a single class-folder directory.

In [ ]:
# ============================================================
# 3. Load Dataset
# ============================================================

def find_split_dir(base, names):
    for name in names:
        p = base / name
        if p.exists() and p.is_dir():
            return p
    return None

train_dir = find_split_dir(DATASET_DIR, ['train', 'training', 'Train', 'Training'])
val_dir = find_split_dir(DATASET_DIR, ['valid', 'validation', 'val', 'Valid', 'Validation', 'Val'])
test_dir = find_split_dir(DATASET_DIR, ['test', 'Test'])

# If DATASET_DIR itself contains train/val folders, use them.
if train_dir is not None and val_dir is not None:
    print('Detected train/validation split.')
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
    if test_dir is not None:
        test_ds = tf.keras.utils.image_dataset_from_directory(
            test_dir,
            seed=SEED,
            image_size=IMG_SIZE,
            batch_size=BATCH_SIZE
        )
    else:
        test_ds = val_ds
else:
    print('Using validation split from one class-folder directory.')
    train_ds = tf.keras.utils.image_dataset_from_directory(
        DATASET_DIR,
        validation_split=0.30,
        subset='training',
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
    temp_ds = tf.keras.utils.image_dataset_from_directory(
        DATASET_DIR,
        validation_split=0.30,
        subset='validation',
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE
    )
    val_size = max(1, int(0.5 * len(temp_ds)))
    val_ds = temp_ds.take(val_size)
    test_ds = temp_ds.skip(val_size)

class_names = train_ds.class_names
num_classes = len(class_names)
print('Classes:', class_names)
print('Number of classes:', num_classes)

In [ ]:
# ============================================================
# 4. Class Distribution and Sample Images
# ============================================================

class_counts = {name: 0 for name in class_names}

# Count images from class folders when possible
for name in class_names:
    possible_dirs = list(DATASET_DIR.rglob(name))
    count = 0
    for d in possible_dirs:
        if d.is_dir():
            count += len([f for f in d.iterdir() if f.suffix.lower() in image_exts])
    class_counts[name] = count

print('Class distribution:', class_counts)

plt.figure(figsize=(8, 5))
plt.bar(class_counts.keys(), class_counts.values())
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png')
plt.show()

plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[int(labels[i])])
        plt.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_images.png')
plt.show()

In [ ]:
# ============================================================
# 5. Dataset Optimization and Augmentation
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10)
], name='data_augmentation')

In [ ]:
# ============================================================
# 6. Custom CNN Model
# ============================================================

cnn_model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Rescaling(1./255),
    data_augmentation,

    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

cnn_history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

In [ ]:
# ============================================================
# 7. Transfer Learning Model - MobileNetV2
# ============================================================

base_model = MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

mobile_model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    data_augmentation,
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

mobile_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

mobile_model.summary()

mobile_history = mobile_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

In [ ]:
# ============================================================
# 8. Evaluation Functions
# ============================================================

def evaluate_model(model, dataset, model_name):
    y_true, y_pred = [], []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    print(f'\nClassification Report - {model_name}')
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{model_name}_confusion_matrix.png')
    plt.show()

    acc = np.mean(np.array(y_true) == np.array(y_pred))
    return acc

cnn_acc = evaluate_model(cnn_model, test_ds, 'Custom_CNN')
mobile_acc = evaluate_model(mobile_model, test_ds, 'MobileNetV2')
print('Custom CNN Test Accuracy:', cnn_acc)
print('MobileNetV2 Test Accuracy:', mobile_acc)

In [ ]:
# ============================================================
# 9. Accuracy and Loss Curves
# ============================================================

def plot_history(history, model_name):
    plt.figure(figsize=(7, 5))
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'Accuracy Curve - {model_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{model_name}_accuracy_curve.png')
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'Loss Curve - {model_name}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{model_name}_loss_curve.png')
    plt.show()

plot_history(cnn_history, 'Custom_CNN')
plot_history(mobile_history, 'MobileNetV2')

In [ ]:
# ============================================================
# 10. Model Comparison and Saving
# ============================================================

comparison = pd.DataFrame({
    'Model': ['Custom CNN', 'MobileNetV2'],
    'Test Accuracy': [cnn_acc, mobile_acc]
})

print(comparison)
comparison.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)

cnn_model.save(OUTPUT_DIR / 'custom_cnn_road_damage.h5')
mobile_model.save(OUTPUT_DIR / 'mobilenetv2_road_damage.h5')

print('Saved outputs to:', OUTPUT_DIR)
print('Important files:')
for f in OUTPUT_DIR.iterdir():
    print('-', f)

## Submission Notes

Upload this notebook to GitHub with the proposal document, README file, dataset link, output figures, and saved model comparison table.